# Smart Insect Identifier - Model Training (Colab)
**Nama:** KhairunNisa

**NPM:** 2308107010074

**Dataset:** Kaggle Insects Dataset (118 Classes)

**Model:** EfficientNetB0 (ImageNet Pretrained)

**Training Strategy:** Transfer Learning + Fine-Tuning + IMPROVED AUGMENTATION

## 0. Persiapan Dataset

In [ ]:
!pip install -q kaggle tensorflow matplotlib seaborn scikit-learn
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
DATASET_PATH = '/content/insects'
os.makedirs(DATASET_PATH, exist_ok=True)
!kaggle datasets download -d baxtiyorbotiraliyev/insects -p {DATASET_PATH} --unzip -q
print('Dataset downloaded successfully')

## 1. Import Library

In [ ]:
import os, json, random, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from sklearn.utils import class_weight
from pathlib import Path
from PIL import Image
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_TL = 50
EPOCHS_FT = 60
MODEL_TL_PATH = '/content/model_efficientnet_TL.keras'
MODEL_FT_PATH = '/content/model_efficientnet_FT.keras'
METADATA_PATH = '/content/model_metadata.json'
print(f'TensorFlow: {tf.__version__}')

## 2. Setup Dataset

In [ ]:
IMAGE_ROOT = Path(DATASET_PATH)
class_dirs = sorted([d for d in IMAGE_ROOT.iterdir() if d.is_dir()])
if len(class_dirs) == 1:
    IMAGE_ROOT = class_dirs[0]
    class_dirs = sorted([d for d in IMAGE_ROOT.iterdir() if d.is_dir()])
CLASS_NAMES = [d.name for d in class_dirs]
NUM_CLASSES = len(CLASS_NAMES)
print(f'Total classes: {NUM_CLASSES}')

## 3. Load and Prepare Data

In [ ]:
CACHE_PATH = '/content/sampled_cache.json'
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        cache = json.load(f)
    all_paths = np.array(cache['paths'])
    all_labels = np.array(cache['labels'])
    print(f'Loaded {len(all_paths):,} images from cache')
else:
    all_ds_raw = tf.keras.utils.image_dataset_from_directory(str(IMAGE_ROOT), seed=SEED, image_size=IMG_SIZE, batch_size=None, label_mode='int', shuffle=True)
    all_paths, all_labels = [], []
    for path, label in all_ds_raw.take(-1):
        all_paths.append(path.numpy().decode())
        all_labels.append(label.numpy())
    all_paths = np.array(all_paths)
    all_labels = np.array(all_labels)
    with open(CACHE_PATH, 'w') as f:
        json.dump({'paths': all_paths.tolist(), 'labels': all_labels.tolist(), 'class_names': CLASS_NAMES}, f)
print(f'Total images: {len(all_paths):,}')

## 4. Sampling and Split

In [ ]:
from collections import defaultdict
MAX_PER_CLASS = 500
grouped = defaultdict(list)
for p, l in zip(all_paths, all_labels):
    grouped[l].append((p, l))
sampled_paths, sampled_labels = [], []
for lbl, items in grouped.items():
    if len(items) > MAX_PER_CLASS:
        items = random.sample(items, MAX_PER_CLASS)
    sampled_paths.extend([p for p, l in items])
    sampled_labels.extend([l for p, l in items])
sampled_paths = np.array(sampled_paths)
sampled_labels = np.array(sampled_labels)
X_trainval, X_test, y_trainval, y_test = train_test_split(sampled_paths, sampled_labels, test_size=0.10, stratify=sampled_labels, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.15/(1-0.10), stratify=y_trainval, random_state=SEED)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

## 5. Data Pipeline with Aggressive Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomFlip('vertical'),
    layers.RandomRotation(0.20),
    layers.RandomZoom(0.20),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1)
])
AUTOTUNE = tf.data.AUTOTUNE
def make_ds(paths, labels, aug=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    def load(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3, try_recover_truncated=True)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        if aug:
            img = data_augmentation(img, training=True)
        img = tf.clip_by_value(img, 0.0, 1.0)
        return img, label
    ds = (ds.map(load, num_parallel_calls=AUTOTUNE).ignore_errors().batch(BATCH_SIZE).prefetch(AUTOTUNE))
    return ds
train_ds = make_ds(X_train, y_train, aug=True, shuffle=True)
val_ds = make_ds(X_val, y_val)
test_ds = make_ds(X_test, y_test)
cw = class_weight.compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weights = dict(enumerate(cw))
print('Data pipeline ready')

## 6. Build Transfer Learning Model

In [ ]:
def build_tl_model(num_classes):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=IMG_SIZE+(3,))
    base_model.trainable = False
    inputs = keras.Input(shape=IMG_SIZE+(3,))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = keras.Model(inputs, outputs, name='EfficientNetB0_TL')
    return model, base_model
model_tl, base_model = build_tl_model(NUM_CLASSES)
model_tl.compile(optimizer=keras.optimizers.Adam(5e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callbacks_tl = [EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1), ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1), ModelCheckpoint(MODEL_TL_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)]

## 7. Train Transfer Learning

In [ ]:
start_tl = time.time()
history_tl = model_tl.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TL, callbacks=callbacks_tl, class_weight=class_weights, verbose=1)
time_tl = time.time() - start_tl
print(f'TL training time: {time_tl/60:.1f} minutes')

## 8. Fine Tuning Setup

In [ ]:
base_model.trainable = True
total_layers = len(base_model.layers)
fine_tune_at = total_layers - 40
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
model_tl.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callbacks_ft = [EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1), ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-8, verbose=1), ModelCheckpoint(MODEL_FT_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)]

## 9. Train Fine Tuning

In [ ]:
start_ft = time.time()
history_ft = model_tl.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, callbacks=callbacks_ft, class_weight=class_weights, verbose=1)
time_ft = time.time() - start_ft
print(f'FT training time: {time_ft/60:.1f} minutes')

## 10. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(history_tl.history['accuracy'], 'b-o', label='Train')
axes[0,0].plot(history_tl.history['val_accuracy'], 'r-o', label='Val')
axes[0,0].set_title('TL Accuracy')
axes[0,0].legend()
axes[0,1].plot(history_tl.history['loss'], 'b-o', label='Train')
axes[0,1].plot(history_tl.history['val_loss'], 'r-o', label='Val')
axes[0,1].set_title('TL Loss')
axes[0,1].legend()
axes[1,0].plot(history_ft.history['accuracy'], 'b-o', label='Train')
axes[1,0].plot(history_ft.history['val_accuracy'], 'r-o', label='Val')
axes[1,0].set_title('FT Accuracy')
axes[1,0].legend()
axes[1,1].plot(history_ft.history['loss'], 'b-o', label='Train')
axes[1,1].plot(history_ft.history['val_loss'], 'r-o', label='Val')
axes[1,1].set_title('FT Loss')
axes[1,1].legend()
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Evaluate Model

In [ ]:
from tensorflow.keras.models import load_model
model_tl_eval = load_model(MODEL_TL_PATH)
model_ft_eval = load_model(MODEL_FT_PATH)
y_true, y_pred_tl, y_pred_ft = [], [], []
for images, labels in test_ds:
    y_true.extend(labels.numpy())
    y_pred_tl.extend(np.argmax(model_tl_eval.predict(images, verbose=0), axis=1))
    y_pred_ft.extend(np.argmax(model_ft_eval.predict(images, verbose=0), axis=1))
y_true = np.array(y_true)
y_pred_tl = np.array(y_pred_tl)
y_pred_ft = np.array(y_pred_ft)

## 12. Calculate Metrics

In [ ]:
def calc_metrics(y_true, y_pred, title):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f'{title}: Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}')
    return acc, prec, rec, f1
tl_acc, tl_prec, tl_rec, tl_f1 = calc_metrics(y_true, y_pred_tl, 'Transfer Learning')
ft_acc, ft_prec, ft_rec, ft_f1 = calc_metrics(y_true, y_pred_ft, 'Fine-Tuning')

## 13. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(28, 12))
for ax, y_p, title in zip(axes, [y_pred_tl, y_pred_ft], ['TL', 'FT']):
    cm = confusion_matrix(y_true, y_p)
    cm_n = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_n, annot=False, fmt='.2f', cmap='Blues', linewidths=0.3, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 14. Export Model and Metadata

In [ ]:
model_tl_eval.save(MODEL_TL_PATH)
model_tl_eval.save('/content/model_efficientnet_TL.h5')
model_ft_eval.save(MODEL_FT_PATH)
model_ft_eval.save('/content/model_efficientnet_FT.h5')
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}
metadata = {
    'class_names': CLASS_NAMES,
    'label2idx': label2idx,
    'num_classes': NUM_CLASSES,
    'img_size': list(IMG_SIZE),
    'model': 'EfficientNetB0',
    'test_accuracy_tl': float(tl_acc),
    'test_accuracy_ft': float(ft_acc),
    'imagenet_mean': [0.485, 0.456, 0.406],
    'imagenet_std': [0.229, 0.224, 0.225]
}
with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)
print('All files exported successfully')